# 02 — Merton jump-diffusion playground (Monte Carlo)

Synthetic paths only — no market data.

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda \kappa)\,dt + \sigma\,dW_t + (e^J - 1)\,dN_t$$

where \(N_t\) is Poisson with intensity \(\lambda\), and \(J \sim \mathcal{N}(\mu_J, \sigma_J^2)\), \(\kappa = \mathbb{E}[e^J-1]\).

Raise \(\lambda\) or \(\sigma_J\) to see fat tails and sudden jumps in the path cloud.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_merton(mu, sigma, lam, mu_j, sigma_j, S0, T, n_steps, n_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0
    z = rng.standard_normal((n_paths, n_steps))
    n_jumps = rng.poisson(lam * dt, size=(n_paths, n_steps))
    jump_sizes = np.zeros_like(z)
    mask = n_jumps > 0
    # compound Poisson: sum of n_jumps normal jumps
    jump_sizes[mask] = (
        n_jumps[mask] * mu_j
        + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
    )
    increments = (mu - 0.5 * sigma**2 - lam * kappa) * dt + sigma * np.sqrt(dt) * z + jump_sizes
    log_paths = np.cumsum(increments, axis=1)
    paths = S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), log_paths]))
    t = np.linspace(0, T, n_steps + 1)
    return t, paths, increments

def plot_merton(
    mu=0.08, sigma=0.18, lam=0.5, mu_j=-0.05, sigma_j=0.10,
    S0=100.0, T=1.0, n_steps=252, n_paths=50,
):
    t, paths, increments = simulate_merton(
        mu, sigma, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.9)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean path")
    axes[0].set_title("Merton Monte Carlo paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].hist(increments.ravel(), bins=80, density=True, alpha=0.75, color="darkorange")
    axes[1].set_title("Step log-return distribution (with jumps)")
    axes[1].set_xlabel("log return")
    axes[1].set_ylabel("density")

    fig.suptitle(
        f"μ={mu:.2f}, σ={sigma:.2f}, λ={lam:.2f}, μJ={mu_j:.2f}, σJ={sigma_j:.2f}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_merton,
    mu=FloatSlider(value=0.08, min=-0.20, max=0.40, step=0.01, description="μ"),
    sigma=FloatSlider(value=0.18, min=0.01, max=0.60, step=0.01, description="σ"),
    lam=FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description="λ jumps/y"),
    mu_j=FloatSlider(value=-0.05, min=-0.40, max=0.20, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="T"),
    n_steps=IntSlider(value=252, min=50, max=1000, step=10, description="steps"),
    n_paths=IntSlider(value=50, min=5, max=200, step=5, description="paths"),
);

interactive(children=(FloatSlider(value=0.08, description='μ', max=0.4, min=-0.2, step=0.01), FloatSlider(valu…